In [ ]:
import psutil

mem = psutil.virtual_memory()

print("Total RAM:", round(mem.total / 1024**3, 2), "GB")
print("Available RAM:", round(mem.available / 1024**3, 2), "GB")

Total RAM: 12.67 GB
Available RAM: 11.7 GB


In [2]:
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id="HayatoHongo/AIkenSGTv1",
    filename="model_clean.safetensors",
)

print(model_path)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


model_clean.safetensors: reconstructing file:   0%|          |  0.00B / 10.5GB            

model_clean.safetensors: downloading bytes:           |  0.00B            

/root/.cache/huggingface/hub/models--HayatoHongo--AIkenSGTv1/snapshots/48915dc0fb07dd5cf43c868dceaa1f9c5ad6c124/model_clean.safetensors


In [ ]:
import os

size_gib = os.path.getsize(model_path) / 1024**3
print(f"model_clean.safetensors: {size_gib:.2f} GiB")

model_clean.safetensors: 9.75 GiB


In [4]:
# added top-p and top-k filtering in generate function
# set vocab_size in config.py
# MHA with KV cache + RoPE + PyTorch SDPA.
# This traditional implementation is easier to understand, and still efficient in practice.
# GQA and MLA is a great way for long-text inference with reduced KV cache size,
# but both comes with slight loss increase and no efficiency merits during training phase.
# KV cache does not help training speed. Codebase will be simpler without it.
# KV cache supports multi-turn continuation by RoPE with position offset.
# No Dropout. Dataset is large enough and regularization is not necessary.

import torch
import torch.nn as nn
import torch.nn.functional as F

class TokenEmbedding(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.token_embedding_table = nn.Embedding(config.vocab_size, config.embedding_dim)
        # keep embedding in default dtype (autocast will handle bf16 when enabled)

    def forward(self, input_indices):
        return self.token_embedding_table(input_indices)


class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=2048, rope_theta=1e6):
        super().__init__()

        inv_freq = 1.0 / (rope_theta ** (torch.arange(0, dim, 2) / dim))
        position_index = torch.arange(max_seq_len)
        frequency_matrix = torch.einsum('i,j->ij', position_index, inv_freq)

        cosine = torch.cos(frequency_matrix)[None, None, :, :]
        sine = torch.sin(frequency_matrix)[None, None, :, :]

        self.register_buffer("cos_cached", cosine, persistent=False)
        self.register_buffer("sin_cached", sine, persistent=False)

    def apply_rotary_emb(self, x, position_offset=0):
        sequence_length = x.size(2)

        cosine = self.cos_cached[:, :, position_offset:position_offset + sequence_length, :]
        sine = self.sin_cached[:, :, position_offset:position_offset + sequence_length, :]

        x_even = x[..., 0::2]
        x_odd = x[..., 1::2]

        rotated_even = x_even * cosine - x_odd * sine
        rotated_odd = x_odd * cosine + x_even * sine

        rotated = torch.empty_like(x)
        rotated[..., 0::2] = rotated_even
        rotated[..., 1::2] = rotated_odd

        return rotated

class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.num_heads = config.num_attention_heads
        self.embed_dim = config.embedding_dim
        self.head_dim = self.embed_dim // self.num_heads

        # QKV projection
        self.query_fc = nn.Linear(self.embed_dim, self.embed_dim, bias=False)
        self.key_fc   = nn.Linear(self.embed_dim, self.embed_dim, bias=False)
        self.value_fc = nn.Linear(self.embed_dim, self.embed_dim, bias=False)

        # Rotary Positional Embedding (RoPE)
        self.rotary_emb = RotaryEmbedding(
            dim=self.head_dim,
            max_seq_len=config.max_sequence_length,
            rope_theta=config.rope_theta
        )

        self.output_projection = nn.Linear(self.embed_dim, self.embed_dim)

        self.register_buffer(
            "causal_mask",
            torch.tril(torch.ones(
                config.max_sequence_length,
                config.max_sequence_length,
                dtype=torch.bool
            )),
            persistent=False
        )

        # KV cache
        self.register_buffer("cache_k", None, persistent=False)
        self.register_buffer("cache_v", None, persistent=False)
        self.current_pos = 0

    # --------------------------------------------------
    # router
    # --------------------------------------------------
    def forward(self, x, use_cache=False):
        input_len = x.size(1)
        if use_cache is False:
            return self.forward_no_cache(x)
        elif use_cache is True and input_len > 1:
            return self.forward_prefill(x)
        elif use_cache is True and input_len == 1: # Hi scenario also starts with T==1
            return self.forward_cached_decoding(x)
        else:
            raise RuntimeError("Unexpected condition in MultiHeadAttention forward")

    # --------------------------------------------------
    # (1) no cache : training 
    # --------------------------------------------------
    def forward_no_cache(self, x):
        B, T, C = x.shape

        Q = self.query_fc(x)
        K = self.key_fc(x)
        V = self.value_fc(x)

        Q = Q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # RoPE : offset = 0
        Q = self.rotary_emb.apply_rotary_emb(Q, position_offset=0)
        K = self.rotary_emb.apply_rotary_emb(K, position_offset=0)

        out = F.scaled_dot_product_attention(
            Q, K, V,
            attn_mask=None,
            is_causal=True
        )

        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.output_projection(out)
        return out

    # --------------------------------------------------
    # (2) prefill : initialize KV cache
    # --------------------------------------------------
    def forward_prefill(self, x):
        B, T, C = x.shape

        Q = self.query_fc(x)
        K = self.key_fc(x)
        V = self.value_fc(x)

        Q = Q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # init cache
        if self.cache_k is None:
            self.cache_k = torch.zeros(
                B, self.num_heads, self.config.max_sequence_length, self.head_dim,
                device=x.device, dtype=K.dtype
            )
            self.cache_v = torch.zeros(
                B, self.num_heads, self.config.max_sequence_length, self.head_dim,
                device=x.device, dtype=V.dtype
            )
            self.current_pos = 0

        # RoPE : offset = current_pos (supports multi-turn continuation)
        Q = self.rotary_emb.apply_rotary_emb(Q, position_offset=self.current_pos)
        K = self.rotary_emb.apply_rotary_emb(K, position_offset=self.current_pos)

        # prevent overflow
        if self.current_pos + T > self.config.max_sequence_length:
            raise RuntimeError("KV cache exceeded max_sequence_length")

        self.cache_k[:, :, self.current_pos:self.current_pos + T, :] = K
        self.cache_v[:, :, self.current_pos:self.current_pos + T, :] = V

        K = self.cache_k[:, :, :self.current_pos + T, :]
        V = self.cache_v[:, :, :self.current_pos + T, :]

        attn_mask = self.causal_mask[
            self.current_pos : self.current_pos + T,
            : self.current_pos + T
        ]

        out = F.scaled_dot_product_attention(
            Q, K, V,
            attn_mask=attn_mask,
            is_causal=False
        )

        self.current_pos += T

        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.output_projection(out)
        return out

    # --------------------------------------------------
    # (3) decode : cached decoding (1 token)
    # --------------------------------------------------
    def forward_cached_decoding(self, x):
        B, T, C = x.shape
        assert T == 1, "cached decoding expects T==1"

        Q = self.query_fc(x)
        K = self.key_fc(x)
        V = self.value_fc(x)

        Q = Q.view(B, 1, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, 1, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, 1, self.num_heads, self.head_dim).transpose(1, 2)

        # This is not usually needed since prefill should have initialized the cache.
        # Just in case for "Hi" scenario, which starts with single token input.
        if self.cache_k is None:
            self.cache_k = torch.zeros(
                B, self.num_heads, self.config.max_sequence_length, self.head_dim,
                device=x.device, dtype=K.dtype
            )
            self.cache_v = torch.zeros(
                B, self.num_heads, self.config.max_sequence_length, self.head_dim,
                device=x.device, dtype=V.dtype
            )
            self.current_pos = 0

        if self.current_pos + 1 >= self.config.max_sequence_length:
            raise RuntimeError("KV cache exceeded max_sequence_length")

        # RoPE : offset = current_pos
        Q = self.rotary_emb.apply_rotary_emb(Q, position_offset=self.current_pos)
        K = self.rotary_emb.apply_rotary_emb(K, position_offset=self.current_pos)

        self.cache_k[:, :, self.current_pos:self.current_pos + 1, :] = K
        self.cache_v[:, :, self.current_pos:self.current_pos + 1, :] = V

        K = self.cache_k[:, :, :self.current_pos + 1, :]
        V = self.cache_v[:, :, :self.current_pos + 1, :]

        out = F.scaled_dot_product_attention(
            Q, K, V,
            attn_mask=None,
            is_causal=False
        )

        self.current_pos += 1

        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.output_projection(out)
        return out

    def reset_cache(self):
        self.cache_k = None
        self.cache_v = None
        self.current_pos = 0



class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()    
        self.net = nn.Sequential(
            nn.Linear(config.embedding_dim, config.hidden_dim, bias=False),
            nn.ReLU(),
            nn.Linear(config.hidden_dim, config.embedding_dim, bias=False),
        )

    def forward(self, input_tensor):
        return self.net(input_tensor)


class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.layer_norm1 = nn.LayerNorm(config.embedding_dim)
        self.layer_norm2 = nn.LayerNorm(config.embedding_dim)
        self.multihead_attention = MultiHeadAttention(config=config)
        self.feed_forward = FeedForward(config=config)


    def forward(self, input_tensor, use_cache=False):
        normed_input = self.layer_norm1(input_tensor)
        attention_output = self.multihead_attention(normed_input, use_cache=use_cache)
        residual_attention = attention_output + input_tensor
        normed_attention = self.layer_norm2(residual_attention)
        feedforward_output = self.feed_forward(normed_attention)
        final_output = feedforward_output + residual_attention
        return final_output


class VocabularyLogits(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.output_norm = nn.LayerNorm(config.embedding_dim)
        self.vocab_projection = nn.Linear(config.embedding_dim, config.vocab_size, bias=False)

    def forward(self, transformer_block_output):
        x = transformer_block_output
        normalized_output = self.output_norm(x)
        vocab_logits = self.vocab_projection(normalized_output)
        return vocab_logits


class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token_embedding_layer = TokenEmbedding(config=config)
        self.blocks = nn.ModuleList([TransformerBlock(config=config) for _ in range(config.layer_count)])
        self.vocab_projection = VocabularyLogits(config=config)
        self.criterion = nn.CrossEntropyLoss()


    def forward(self, input_indices, target_indices, use_cache=False):
        token_embeddings = self.token_embedding_layer.forward(input_indices)

        x = token_embeddings
        for block in self.blocks:
            x = block(x, use_cache=use_cache)
        logits = self.vocab_projection(x)

        if target_indices is None:
            return logits, None

        batch_size, token_len, vocab_size = logits.shape
        logits_flat = logits.view(batch_size * token_len, vocab_size)
        targets_flat = target_indices.view(batch_size * token_len)
        loss = self.criterion(logits_flat, targets_flat)
        return logits, loss


    def generate(self,
        input_indices,
        max_new_tokens,
        temperature=1.0,
        use_cache=True,
        reset_cache=False,
        top_k=None,      # ### NEW ###
        top_p=None,      # ### NEW ###
    ):
        self.eval()

        if reset_cache:
            for block in self.blocks:
                block.multihead_attention.reset_cache()

        next_token = None

        for i in range(max_new_tokens):
            if use_cache:
                if i == 0:
                    logits, _ = self.forward(input_indices, None, use_cache=True)
                else:
                    logits, _ = self.forward(next_token, None, use_cache=True)
            else:
                logits, _ = self.forward(input_indices, None, use_cache=False)

            """ DELETE
            last_logits = logits[:, -1, :] / temperature
            probs = F.softmax(last_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            """

            ### NEW ###
            last_logits = logits[:, -1, :] / temperature

            if top_k is not None:
                top_k = min(top_k, last_logits.size(-1))
                values, _ = torch.topk(last_logits, top_k)
                min_value = values[:, -1].unsqueeze(-1)
                last_logits = torch.where(
                    last_logits < min_value,
                    torch.full_like(last_logits, float("-inf")),
                    last_logits,
                )

            if top_p is not None:
                sorted_logits, sorted_indices = torch.sort(last_logits, descending=True)
                sorted_probs = F.softmax(sorted_logits, dim=-1)
                cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

                sorted_mask = cumulative_probs > top_p
                sorted_mask[..., 1:] = sorted_mask[..., :-1].clone()
                sorted_mask[..., 0] = False

                sorted_logits = torch.where(
                    sorted_mask,
                    torch.full_like(sorted_logits, float("-inf")),
                    sorted_logits,
                )

                last_logits = torch.zeros_like(last_logits).scatter(
                    -1, sorted_indices, sorted_logits
                )

            probs = F.softmax(last_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            ### NEW ###

            yield int(next_token.item())
            input_indices = torch.cat((input_indices, next_token), dim=1)

In [5]:
class Config:
    embedding_dim: int = 2560
    hidden_dim: int = 10240
    num_attention_heads: int = 20
    layer_count: int = 30
    rope_theta: float = 1_000_000.0
    vocab_size: int = 50257
    max_sequence_length: int = 2048

In [6]:
import gc
import torch
from safetensors import safe_open

gc.collect()
torch.cuda.empty_cache()

device = torch.device("cuda")
config = Config()

print("GPU:", torch.cuda.get_device_name(0))
print(
    "Before model:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GiB"
)

# CPUではなく、最初からGPU上にFP32モデルを構築する
with torch.device(device):
    model = GPT(config)

model.eval()

print(
    "After model creation:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GiB"
)

GPU: Tesla T4
Before model: 0.0 GiB
After model creation: 10.01 GiB


In [7]:
from safetensors import safe_open

param_map = dict(model.named_parameters())

with safe_open(model_path, framework="pt", device="cpu") as f:
    checkpoint_keys = set(f.keys())
    model_keys = set(param_map.keys())

    missing = model_keys - checkpoint_keys
    unexpected = checkpoint_keys - model_keys

    print("Missing keys:", missing)
    print("Unexpected keys:", unexpected)

    if missing or unexpected:
        raise RuntimeError("Checkpoint keys do not match model parameters.")

    with torch.no_grad():
        total = len(checkpoint_keys)

        for i, name in enumerate(f.keys(), 1):
            tensor = f.get_tensor(name)
            param_map[name].copy_(tensor)
            del tensor

            if i % 20 == 0 or i == total:
                print(f"Loaded {i}/{total}")

print("Weights loaded successfully.")
print(
    "GPU memory:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GiB"
)

Missing keys: set()
Unexpected keys: set()
Loaded 20/334
Loaded 40/334
Loaded 60/334
Loaded 80/334
Loaded 100/334
Loaded 120/334
Loaded 140/334
Loaded 160/334
Loaded 180/334
Loaded 200/334
Loaded 220/334
Loaded 240/334
Loaded 260/334
Loaded 280/334
Loaded 300/334
Loaded 320/334
Loaded 334/334
Weights loaded successfully.
GPU memory: 10.01 GiB


In [8]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

In [9]:
prompt = "The future of artificial intelligence is"

input_indices = torch.tensor(
    [tokenizer.encode(prompt)],
    dtype=torch.long,
    device=device
)

In [10]:
with torch.no_grad():
    generated_ids = list(
        model.generate(
            input_indices,
            max_new_tokens=50,
            temperature=0.8,
            top_k=50,
            top_p=0.9,
            reset_cache=True
        )
    )

print(prompt + tokenizer.decode(generated_ids))

The future of artificial intelligence is bound to be far more advanced than that of the human race, and a lot of people are in a position to help. In this article, we will take a look at what AI is, how it works, how it is used in real-


In [11]:
LABELS = ["A", "B", "C", "D"]

answer_token_ids = []

for label in LABELS:
    ids = tokenizer.encode(" " + label)
    print(repr(" " + label), ids)

    assert len(ids) == 1, f"{label} is not a single token"
    answer_token_ids.append(ids[0])

print("Answer token IDs:", answer_token_ids)

' A' [317]
' B' [347]
' C' [327]
' D' [360]
Answer token IDs: [317, 347, 327, 360]


In [12]:
import torch.nn.functional as F

@torch.inference_mode()
def get_choice_probs(prompt):
    token_ids = tokenizer.encode(prompt)

    if len(token_ids) > config.max_sequence_length:
        raise ValueError(
            f"Prompt too long: {len(token_ids)} > {config.max_sequence_length}"
        )

    input_indices = torch.tensor(
        [token_ids],
        dtype=torch.long,
        device=device
    )

    # generate() は使わず、直接 logits を取得
    logits, _ = model(
        input_indices,
        None,
        use_cache=False
    )

    # プロンプト末尾の次に来るtokenのlogits
    next_token_logits = logits[0, -1, :]

    # " A", " B", " C", " D" のlogitsだけ取得
    choice_logits = next_token_logits[answer_token_ids]

    # 4択内でsoftmax
    choice_probs = F.softmax(
        choice_logits.float(),
        dim=-1
    )

    return {
        label: float(prob)
        for label, prob in zip(LABELS, choice_probs)
    }

In [13]:
CYCLIC_PERMUTATIONS = [
    [0, 1, 2, 3],  # A B C D
    [1, 2, 3, 0],  # B C D A
    [2, 3, 0, 1],  # C D A B
    [3, 0, 1, 2],  # D A B C
]

def make_mmlu_prompt(question, choices, permutation):
    lines = [f"Question: {question}"]

    for displayed_idx, original_idx in enumerate(permutation):
        displayed_label = LABELS[displayed_idx]
        lines.append(
            f"{displayed_label}. {choices[original_idx]}"
        )

    lines.append("Answer:")
    return "\n".join(lines)


def evaluate_with_position_debiasing(question, choices):
    semantic_scores = [0.0, 0.0, 0.0, 0.0]
    permutation_results = []

    for perm_idx, permutation in enumerate(CYCLIC_PERMUTATIONS):
        prompt = make_mmlu_prompt(
            question,
            choices,
            permutation
        )

        displayed_probs = get_choice_probs(prompt)

        # 表示上のA/B/C/Dの確率を、
        # 元の意味上の選択肢へ戻す
        semantic_probs = [0.0, 0.0, 0.0, 0.0]

        for displayed_idx, original_idx in enumerate(permutation):
            displayed_label = LABELS[displayed_idx]
            prob = displayed_probs[displayed_label]

            semantic_probs[original_idx] = prob
            semantic_scores[original_idx] += prob

        permutation_results.append({
            "permutation": permutation,
            "displayed_probs": displayed_probs,
            "semantic_probs": semantic_probs,
        })

    # 4配置の平均
    semantic_scores = [
        score / len(CYCLIC_PERMUTATIONS)
        for score in semantic_scores
    ]

    # baseline = 元の配置（最初の permutation）
    baseline_probs = permutation_results[0]["semantic_probs"]
    baseline_idx = max(
        range(4),
        key=lambda i: baseline_probs[i]
    )

    # debiased = 4配置平均
    debiased_idx = max(
        range(4),
        key=lambda i: semantic_scores[i]
    )

    return {
        "baseline_pred": LABELS[baseline_idx],
        "baseline_probs": baseline_probs,
        "debiased_pred": LABELS[debiased_idx],
        "debiased_scores": semantic_scores,
        "permutation_results": permutation_results,
    }

In [14]:
question = "What is the capital of France?"

choices = [
    "Berlin",
    "Madrid",
    "Paris",
    "Rome",
]

result = evaluate_with_position_debiasing(
    question,
    choices
)

print("Baseline prediction:", result["baseline_pred"])
print("Debiased prediction:", result["debiased_pred"])

print("\nBaseline probabilities:")
for label, score in zip(LABELS, result["baseline_probs"]):
    print(f"{label}: {score:.6f}")

print("\nDebiased scores:")
for label, score in zip(LABELS, result["debiased_scores"]):
    print(f"{label}: {score:.6f}")

print("\nEach permutation:")
for i, r in enumerate(result["permutation_results"]):
    print(
        i,
        r["permutation"],
        r["displayed_probs"]
    )

Baseline prediction: C
Debiased prediction: C

Baseline probabilities:
A: 0.188010
B: 0.282342
C: 0.322167
D: 0.207480

Debiased scores:
A: 0.226263
B: 0.262557
C: 0.264561
D: 0.246620

Each permutation:
0 [0, 1, 2, 3] {'A': 0.18801045417785645, 'B': 0.28234240412712097, 'C': 0.32216697931289673, 'D': 0.20748023688793182}
1 [1, 2, 3, 0] {'A': 0.25296688079833984, 'B': 0.26187077164649963, 'C': 0.3213290870189667, 'D': 0.16383330523967743}
2 [2, 3, 0, 1] {'A': 0.24536527693271637, 'B': 0.2643398642539978, 'C': 0.266013503074646, 'D': 0.22428135573863983}
3 [3, 0, 1, 2] {'A': 0.19332893192768097, 'B': 0.2871949374675751, 'C': 0.290636271238327, 'D': 0.2288399040699005}


In [23]:
!unzip -q /content/mmlu_data.zip -d /content
!ls /content/mmlu-reference/data | head

dev
test


In [24]:
!ls /content/mmlu-reference/data/test | head

abstract_algebra_test.csv
anatomy_test.csv
astronomy_test.csv
business_ethics_test.csv
clinical_knowledge_test.csv
college_biology_test.csv
college_chemistry_test.csv
college_computer_science_test.csv
college_mathematics_test.csv
college_medicine_test.csv


In [26]:
import pandas as pd

test_path = "/content/mmlu-reference/data/test/abstract_algebra_test.csv"

df = pd.read_csv(test_path, header=None)

print(df.head())
print("shape:", df.shape)

                                                   0           1  \
0  Find the degree for the given field extension ...           0   
1  Let p = (1, 2, 5, 4)(2, 3) in S_5 . Find the i...           8   
2  Find all zeros in the indicated finite field o...           0   
3  Statement 1 | A factor group of a non-Abelian ...  True, True   
4  Find the product of the given polynomials in t...    2x^2 + 5   

               2            3            4  5  
0              4            2            6  B  
1              2           24          120  C  
2              1          0,1          0,4  D  
3   False, False  True, False  False, True  B  
4  6x^2 + 4x + 6            0      x^2 + 1  B  
shape: (100, 6)


In [27]:
row = df.iloc[0]

print("Question:", row[0])
print("A:", row[1])
print("B:", row[2])
print("C:", row[3])
print("D:", row[4])
print("Answer:", row[5])

Question: Find the degree for the given field extension Q(sqrt(2), sqrt(3), sqrt(18)) over Q.
A: 0
B: 4
C: 2
D: 6
Answer: B


In [28]:
question = row[0]

choices = [
    row[1],
    row[2],
    row[3],
    row[4],
]

label = row[5]

result = evaluate_with_position_debiasing(
    question,
    choices
)

print("Question:")
print(question)

print("\nChoices:")
for l, c in zip(LABELS, choices):
    print(f"{l}. {c}")

print("\nCorrect answer:", label)

print("\nBaseline prediction:", result["baseline_pred"])
print("Debiased prediction:", result["debiased_pred"])

print("\nBaseline probabilities:")
for l, score in zip(LABELS, result["baseline_probs"]):
    print(f"{l}: {score:.6f}")

print("\nDebiased scores:")
for l, score in zip(LABELS, result["debiased_scores"]):
    print(f"{l}: {score:.6f}")

print("\nBaseline correct:",
      result["baseline_pred"] == label)

print("Debiased correct:",
      result["debiased_pred"] == label)

Question:
Find the degree for the given field extension Q(sqrt(2), sqrt(3), sqrt(18)) over Q.

Choices:
A. 0
B. 4
C. 2
D. 6

Correct answer: B

Baseline prediction: C
Debiased prediction: A

Baseline probabilities:
A: 0.304008
B: 0.212327
C: 0.319221
D: 0.164443

Debiased scores:
A: 0.265783
B: 0.251128
C: 0.243302
D: 0.239786

Baseline correct: False
Debiased correct: False


In [29]:
import pandas as pd
from pathlib import Path

results = []

for question_idx in range(10):
    row = df.iloc[question_idx]

    question = str(row[0])
    choices = [
        str(row[1]),
        str(row[2]),
        str(row[3]),
        str(row[4]),
    ]
    label = str(row[5]).strip()

    result = evaluate_with_position_debiasing(
        question,
        choices
    )

    baseline_pred = result["baseline_pred"]
    debiased_pred = result["debiased_pred"]

    results.append({
        "subject": "abstract_algebra",
        "question_index": question_idx,
        "label": label,

        "baseline_pred": baseline_pred,
        "baseline_correct": baseline_pred == label,

        "debiased_pred": debiased_pred,
        "debiased_correct": debiased_pred == label,

        "baseline_A": result["baseline_probs"][0],
        "baseline_B": result["baseline_probs"][1],
        "baseline_C": result["baseline_probs"][2],
        "baseline_D": result["baseline_probs"][3],

        "debiased_A": result["debiased_scores"][0],
        "debiased_B": result["debiased_scores"][1],
        "debiased_C": result["debiased_scores"][2],
        "debiased_D": result["debiased_scores"][3],
    })

    print(
        f"{question_idx + 1}/10 "
        f"baseline={baseline_pred} "
        f"debiased={debiased_pred} "
        f"label={label}"
    )

results_df = pd.DataFrame(results)

output_path = "/content/aikengpt_mmlu_test10.csv"
results_df.to_csv(output_path, index=False)

print("\nSaved:", output_path)

print(
    "Baseline accuracy:",
    results_df["baseline_correct"].mean()
)

print(
    "Debiased accuracy:",
    results_df["debiased_correct"].mean()
)

1/10 baseline=C debiased=A label=B
2/10 baseline=A debiased=D label=C
3/10 baseline=A debiased=C label=D
4/10 baseline=B debiased=B label=B
5/10 baseline=C debiased=D label=B
6/10 baseline=B debiased=B label=A
7/10 baseline=B debiased=B label=A
8/10 baseline=B debiased=B label=D
9/10 baseline=C debiased=A label=B
10/10 baseline=A debiased=A label=C

Saved: /content/aikengpt_mmlu_test10.csv
Baseline accuracy: 0.1
Debiased accuracy: 0.1


In [30]:
results_df

,subject,question_index,label,baseline_pred,baseline_correct,debiased_pred,debiased_correct,baseline_A,baseline_B,baseline_C,baseline_D,debiased_A,debiased_B,debiased_C,debiased_D
0,abstract_algebra,0,B,C,False,A,False,0.304008,0.212327,0.319221,0.164443,0.265783,0.251128,0.243302,0.239786
1,abstract_algebra,1,C,A,False,D,False,0.301643,0.278990,0.247945,0.171421,0.237758,0.255436,0.243632,0.263175
2,abstract_algebra,2,D,A,False,C,False,0.412363,0.158452,0.282805,0.146380,0.241621,0.257261,0.258510,0.242607
3,abstract_algebra,3,B,B,True,B,True,0.246750,0.406021,0.180890,0.166339,0.244087,0.260158,0.241122,0.254633
4,abstract_algebra,4,B,C,False,D,False,0.227563,0.143192,0.366245,0.263001,0.232280,0.256818,0.251318,0.259584
5,abstract_algebra,5,A,B,False,B,False,0.188208,0.300396,0.219471,0.291925,0.238861,0.269414,0.232871,0.258855
6,abstract_algebra,6,A,B,False,B,False,0.196358,0.359985,0.224550,0.219107,0.232898,0.268456,0.233693,0.264952
7,abstract_algebra,7,D,B,False,B,False,0.195972,0.368869,0.182758,0.252401,0.240472,0.268734,0.230020,0.260774
8,abstract_algebra,8,B,C,False,A,False,0.309205,0.195414,0.328437,0.166944,0.262680,0.247692,0.245527,0.244101
9,abstract_algebra,9,C,A,False,A,False,0.421514,0.176021,0.263168,0.139297,0.273795,0.252468,0.245846,0.227892


In [31]:
import numpy as np
import torch
import torch.nn.functional as F

LABELS = ["A", "B", "C", "D"]


def format_subject(subject):
    return " " + " ".join(subject.split("_"))


def cyclic_order(shift):
    return [
        (i + shift) % 4
        for i in range(4)
    ]


def format_mmlu_example(
    test_df,
    test_index,
    order=None,
):
    """
    GPT版の format_example(..., include_answer=False)
    と同じ形式。
    """

    if order is None:
        order = [0, 1, 2, 3]

    row = test_df.iloc[test_index]

    prompt = str(row.iloc[0])

    for position, original_index in enumerate(order):
        prompt += "\n{}. {}".format(
            LABELS[position],
            row.iloc[original_index + 1],
        )

    prompt += "\nAnswer:"

    return prompt


def build_mmlu_prompt(
    subject,
    test_df,
    test_index,
    order=None,
):
    """
    ntrain=0 の GPT版と同じプロンプト。
    """

    subject_prompt = (
        "The following are multiple choice questions "
        "(with answers) about{}.\n\n"
    ).format(format_subject(subject))

    return (
        subject_prompt
        + format_mmlu_example(
            test_df,
            test_index,
            order=order,
        )
    )

In [32]:
@torch.inference_mode()
def get_choice_probs_array(prompt):
    token_ids = tokenizer.encode(prompt)

    if len(token_ids) > config.max_sequence_length:
        raise ValueError(
            f"Prompt too long: "
            f"{len(token_ids)} > {config.max_sequence_length}"
        )

    input_indices = torch.tensor(
        [token_ids],
        dtype=torch.long,
        device=device,
    )

    logits, _ = model(
        input_indices,
        None,
        use_cache=False,
    )

    next_logits = logits[0, -1, :]

    choice_logits = next_logits[
        answer_token_ids
    ]

    probs = F.softmax(
        choice_logits.float(),
        dim=-1,
    )

    return probs.cpu().numpy()

In [33]:
def evaluate_mmlu_question(
    subject,
    test_index,
    test_df,
):
    permutation_probs = []
    mapped_probs = []

    for shift in range(4):
        order = cyclic_order(shift)

        prompt = build_mmlu_prompt(
            subject,
            test_df,
            test_index,
            order=order,
        )

        # 表示位置 A/B/C/D の確率
        position_probs = get_choice_probs_array(
            prompt
        )

        permutation_probs.append(
            position_probs
        )

        # semantic optionへ戻す
        mapped = np.zeros(4)

        for position, original_index in enumerate(order):
            mapped[original_index] = (
                position_probs[position]
            )

        mapped_probs.append(mapped)

    # permutation 0 = 通常評価
    baseline_probs = permutation_probs[0]

    baseline_pred = LABELS[
        np.argmax(baseline_probs)
    ]

    # 4配置をsemantic optionに戻して平均
    debiased_probs = np.mean(
        np.stack(mapped_probs),
        axis=0,
    )

    debiased_pred = LABELS[
        np.argmax(debiased_probs)
    ]

    row = test_df.iloc[test_index]

    label = str(
        row.iloc[-1]
    ).strip()

    result = {
        "subject": subject,
        "test_index": int(test_index),

        "question": row.iloc[0],
        "A": row.iloc[1],
        "B": row.iloc[2],
        "C": row.iloc[3],
        "D": row.iloc[4],

        "label": label,

        "baseline_pred": baseline_pred,
        "baseline_correct": (
            baseline_pred == label
        ),

        "debiased_pred": debiased_pred,
        "debiased_correct": (
            debiased_pred == label
        ),
    }

    # baseline probabilities
    for j, label_name in enumerate(LABELS):
        result[
            f"baseline_{label_name}_prob"
        ] = float(baseline_probs[j])

    # debiased probabilities
    for j, label_name in enumerate(LABELS):
        result[
            f"debiased_{label_name}_prob"
        ] = float(debiased_probs[j])

    # 各 permutation の「表示位置」確率
    for shift in range(4):
        for j, label_name in enumerate(LABELS):
            result[
                f"perm{shift}_{label_name}_prob"
            ] = float(
                permutation_probs[shift][j]
            )

    return result

In [36]:
import pandas as pd

manifest_path = "/content/sample_manifest_seed42_frac0p1_all.csv"

manifest = pd.read_csv(manifest_path)

print("Number of rows:", len(manifest))
print(manifest.head())

Number of rows: 1404
                               subject  test_index
0           high_school_macroeconomics         237
1  high_school_government_and_politics         143
2                high_school_chemistry          72
3                            sociology         154
4  high_school_government_and_politics         140


In [37]:
import os
import pandas as pd

DATA_DIR = "/content/mmlu-reference/data"

test_cache = {}
test_results = []

for n, manifest_row in (
    manifest.head(10).iterrows()
):
    subject = str(
        manifest_row["subject"]
    )

    test_index = int(
        manifest_row["test_index"]
    )

    # subjectごとにCSVを一度だけ読む
    if subject not in test_cache:
        path = os.path.join(
            DATA_DIR,
            "test",
            subject + "_test.csv",
        )

        test_cache[subject] = pd.read_csv(
            path,
            header=None,
        )

    test_df = test_cache[subject]

    result = evaluate_mmlu_question(
        subject,
        test_index,
        test_df,
    )

    test_results.append(result)

    print(
        f"{len(test_results)}/10 "
        f"{subject}[{test_index}] "
        f"baseline={result['baseline_pred']} "
        f"debiased={result['debiased_pred']} "
        f"label={result['label']}"
    )

1/10 high_school_macroeconomics[237] baseline=A debiased=A label=A
2/10 high_school_government_and_politics[143] baseline=A debiased=D label=A
3/10 high_school_chemistry[72] baseline=A debiased=A label=D
4/10 sociology[154] baseline=B debiased=A label=D
5/10 high_school_government_and_politics[140] baseline=A debiased=B label=A
6/10 security_studies[62] baseline=A debiased=C label=D
7/10 moral_disputes[95] baseline=A debiased=B label=A
8/10 miscellaneous[455] baseline=B debiased=B label=B
9/10 miscellaneous[87] baseline=C debiased=A label=D
10/10 high_school_biology[58] baseline=A debiased=A label=C


In [38]:
test_results_df = pd.DataFrame(
    test_results
)

print()
print(
    "Baseline accuracy:",
    test_results_df[
        "baseline_correct"
    ].mean()
)

print(
    "Debiased accuracy:",
    test_results_df[
        "debiased_correct"
    ].mean()
)


Baseline accuracy: 0.5
Debiased accuracy: 0.2


In [39]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [40]:
from pathlib import Path

OUTPUT_DIR = Path("/content/drive/MyDrive/aikengpt_mmlu")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "aikengpt_mmlu_10pct_seed42.csv"

print("Output:", OUTPUT_PATH)

Output: /content/drive/MyDrive/aikengpt_mmlu/aikengpt_mmlu_10pct_seed42.csv


In [42]:
print("OUTPUT_DIR :", OUTPUT_DIR)
print("OUTPUT_PATH:", OUTPUT_PATH)
print("Folder exists:", OUTPUT_DIR.exists())
print("CSV exists:", OUTPUT_PATH.exists())

OUTPUT_DIR : /content/drive/MyDrive/aikengpt_mmlu
OUTPUT_PATH: /content/drive/MyDrive/aikengpt_mmlu/aikengpt_mmlu_10pct_seed42.csv
Folder exists: True
CSV exists: True


In [43]:
import pandas as pd

test_save_path = OUTPUT_DIR / "save_test.csv"

pd.DataFrame([
    {"test": 1, "message": "hello"}
]).to_csv(
    test_save_path,
    index=False
)

print("Saved:", test_save_path)
print("Exists:", test_save_path.exists())

Saved: /content/drive/MyDrive/aikengpt_mmlu/save_test.csv
Exists: True


In [44]:
manifest_row = manifest.iloc[0]

subject = str(manifest_row["subject"])
test_index = int(manifest_row["test_index"])

print("Target:", subject, test_index)

test_path = os.path.join(
    DATA_DIR,
    "test",
    f"{subject}_test.csv",
)

test_df = pd.read_csv(
    test_path,
    header=None,
)

print("Evaluating...")

result = evaluate_mmlu_question(
    subject=subject,
    test_index=test_index,
    test_df=test_df,
)

print(
    "Result:",
    result["baseline_pred"],
    result["debiased_pred"],
    result["label"]
)

print("Saving...")

pd.DataFrame([result]).to_csv(
    OUTPUT_PATH,
    index=False,
)

print("Saved:", OUTPUT_PATH)
print("CSV exists:", OUTPUT_PATH.exists())

Target: high_school_macroeconomics 237
Evaluating...
Result: A A A
Saving...
Saved: /content/drive/MyDrive/aikengpt_mmlu/aikengpt_mmlu_10pct_seed42.csv
CSV exists: True


In [46]:
import os
import pandas as pd

DATA_DIR = "/content/mmlu-reference/data"

# subjectごとのCSVを再利用
test_cache = {}

# 既に保存済みの問題を取得
if OUTPUT_PATH.exists():
    existing_df = pd.read_csv(OUTPUT_PATH)

    completed = set(
        zip(
            existing_df["subject"].astype(str),
            existing_df["test_index"].astype(int),
        )
    )

    print(f"Already completed: {len(completed)}")
else:
    completed = set()
    print("Starting from scratch.")

TOTAL = len(manifest)

for _, manifest_row in manifest.iterrows():

    subject = str(manifest_row["subject"])
    test_index = int(manifest_row["test_index"])

    key = (subject, test_index)

    # 評価済みなら飛ばす
    if key in completed:
        continue

    # subjectのCSVを必要なときだけ読む
    if subject not in test_cache:
        test_path = os.path.join(
            DATA_DIR,
            "test",
            f"{subject}_test.csv",
        )

        test_cache[subject] = pd.read_csv(
            test_path,
            header=None,
        )

    test_df = test_cache[subject]

    print(
        f"Evaluating {len(completed)+1}/{TOTAL}: "
        f"{subject}[{test_index}]",
        flush=True,
    )

    try:
        result = evaluate_mmlu_question(
            subject=subject,
            test_index=test_index,
            test_df=test_df,
        )

        # 1問ごとに追記
        pd.DataFrame([result]).to_csv(
            OUTPUT_PATH,
            mode="a",
            header=False,
            index=False,
        )

        completed.add(key)

        print(
            f"  -> baseline={result['baseline_pred']} "
            f"debiased={result['debiased_pred']} "
            f"label={result['label']} "
            f"saved ({len(completed)}/{TOTAL})",
            flush=True,
        )

    except Exception as e:
        print()
        print(
            f"ERROR: {subject}[{test_index}]"
        )
        print(repr(e))
        raise

Already completed: 1
Evaluating 2/1404: high_school_government_and_politics[143]


  -> baseline=A debiased=D label=A saved (2/1404)
Evaluating 3/1404: high_school_chemistry[72]
  -> baseline=A debiased=A label=D saved (3/1404)
Evaluating 4/1404: sociology[154]
  -> baseline=B debiased=A label=D saved (4/1404)
Evaluating 5/1404: high_school_government_and_politics[140]
  -> baseline=A debiased=B label=A saved (5/1404)
Evaluating 6/1404: security_studies[62]
  -> baseline=A debiased=C label=D saved (6/1404)
Evaluating 7/1404: moral_disputes[95]
  -> baseline=A debiased=B label=A saved (7/1404)
Evaluating 8/1404: miscellaneous[455]
  -> baseline=B debiased=B label=B saved (8/1404)
Evaluating 9/1404: miscellaneous[87]
  -> baseline=C debiased=A label=D saved (9/1404)
Evaluating 10/1404: high_school_biology[58]
  -> baseline=A debiased=A label=C saved (10/1404)
Evaluating 11/1404: moral_scenarios[876]
  -> baseline=B debiased=C label=B saved (11/1404)
Evaluating 12/1404: professional_psychology[10]
  -> baseline=A debiased=B label=B saved (12/1404)
Evaluating 13/1404: lo

In [47]:
import pandas as pd

final_df = pd.read_csv(OUTPUT_PATH)

print("Rows:", len(final_df))
print(
    "Unique questions:",
    final_df[["subject", "test_index"]]
    .drop_duplicates()
    .shape[0]
)

print("Subjects:", final_df["subject"].nunique())
print("Missing values:", final_df.isna().sum().sum())

Rows: 1404
Unique questions: 1404
Subjects: 57
Missing values: 0


In [48]:
baseline_acc = final_df["baseline_correct"].mean()
debiased_acc = final_df["debiased_correct"].mean()

changed = (
    final_df["baseline_pred"]
    != final_df["debiased_pred"]
)

print(f"Baseline accuracy: {baseline_acc:.4f} ({baseline_acc*100:.2f}%)")
print(f"Debiased accuracy: {debiased_acc:.4f} ({debiased_acc*100:.2f}%)")

print(
    "Prediction changed:",
    changed.sum(),
    f"({changed.mean()*100:.2f}%)"
)

print(
    "Baseline correct:",
    final_df["baseline_correct"].sum(),
    "/",
    len(final_df)
)

print(
    "Debiased correct:",
    final_df["debiased_correct"].sum(),
    "/",
    len(final_df)
)

Baseline accuracy: 0.2607 (26.07%)
Debiased accuracy: 0.2557 (25.57%)
Prediction changed: 1004 (71.51%)
Baseline correct: 366 / 1404
Debiased correct: 359 / 1404


In [49]:
became_correct = (
    (~final_df["baseline_correct"])
    & final_df["debiased_correct"]
)

became_wrong = (
    final_df["baseline_correct"]
    & (~final_df["debiased_correct"])
)

print("Wrong -> Correct:", became_correct.sum())
print("Correct -> Wrong:", became_wrong.sum())
print("Net gain:", became_correct.sum() - became_wrong.sum())

Wrong -> Correct: 241
Correct -> Wrong: 248
Net gain: -7


In [50]:
for label in ["A", "B", "C", "D"]:
    cols = [
        f"perm0_{label}_prob",
        f"perm1_{label}_prob",
        f"perm2_{label}_prob",
        f"perm3_{label}_prob",
    ]

    mean_prob = final_df[cols].to_numpy().mean()

    print(
        f"Displayed position {label}: "
        f"{mean_prob:.6f}"
    )

Displayed position A: 0.327497
Displayed position B: 0.259410
Displayed position C: 0.200281
Displayed position D: 0.212813


In [51]:
subject_results = (
    final_df
    .groupby("subject")
    .agg(
        n=("test_index", "size"),
        baseline_accuracy=("baseline_correct", "mean"),
        debiased_accuracy=("debiased_correct", "mean"),
    )
)

subject_results["difference"] = (
    subject_results["debiased_accuracy"]
    - subject_results["baseline_accuracy"]
)

subject_results = subject_results.sort_values(
    "difference",
    ascending=False
)

subject_results.to_csv(
    OUTPUT_DIR / "aikengpt_mmlu_10pct_by_subject.csv"
)

subject_results

,n,baseline_accuracy,debiased_accuracy,difference
subject,,,,
high_school_us_history,22,0.272727,0.500000,0.227273
clinical_knowledge,29,0.206897,0.413793,0.206897
us_foreign_policy,15,0.200000,0.400000,0.200000
high_school_physics,12,0.083333,0.250000,0.166667
formal_logic,12,0.166667,0.333333,0.166667
professional_psychology,68,0.176471,0.338235,0.161765
anatomy,14,0.214286,0.357143,0.142857
professional_medicine,28,0.142857,0.285714,0.142857
college_biology,8,0.375000,0.500000,0.125000


In [52]:
NTRAIN = 5

def format_dev_example(dev_df, index):
    row = dev_df.iloc[index]

    prompt = str(row.iloc[0])

    for i, label in enumerate(LABELS):
        prompt += f"\n{label}. {row.iloc[i + 1]}"

    # dev例では正解まで見せる
    answer = str(row.iloc[-1]).strip()
    prompt += f"\nAnswer: {answer}\n\n"

    return prompt


def build_mmlu_prompt_5shot(
    subject,
    dev_df,
    test_df,
    test_index,
    order=None,
    ntrain=5,
):
    if order is None:
        order = [0, 1, 2, 3]

    prompt = (
        "The following are multiple choice questions "
        "(with answers) about{}.\n\n"
    ).format(format_subject(subject))

    # devの先頭5問をdemonstrationとして追加
    for i in range(ntrain):
        prompt += format_dev_example(dev_df, i)

    # 評価対象だけ、必要に応じて選択肢を並べ替える
    prompt += format_mmlu_example(
        test_df,
        test_index,
        order=order,
    )

    return prompt

In [53]:
import os
import pandas as pd
import numpy as np

DATA_DIR = "/content/mmlu-reference/data"

dev_cache = {}
test_cache = {}

length_records = []

for _, manifest_row in manifest.iterrows():
    subject = str(manifest_row["subject"])
    test_index = int(manifest_row["test_index"])

    if subject not in dev_cache:
        dev_path = os.path.join(
            DATA_DIR,
            "dev",
            f"{subject}_dev.csv",
        )
        dev_cache[subject] = pd.read_csv(
            dev_path,
            header=None,
        )

    if subject not in test_cache:
        test_path = os.path.join(
            DATA_DIR,
            "test",
            f"{subject}_test.csv",
        )
        test_cache[subject] = pd.read_csv(
            test_path,
            header=None,
        )

    dev_df = dev_cache[subject]
    test_df = test_cache[subject]

    lengths = []

    # 念のため4 permutationすべて検査
    for shift in range(4):
        order = cyclic_order(shift)

        prompt = build_mmlu_prompt_5shot(
            subject=subject,
            dev_df=dev_df,
            test_df=test_df,
            test_index=test_index,
            order=order,
            ntrain=5,
        )

        lengths.append(
            len(tokenizer.encode(prompt))
        )

    length_records.append({
        "subject": subject,
        "test_index": test_index,
        "min_tokens": min(lengths),
        "max_tokens": max(lengths),
    })

length_df = pd.DataFrame(length_records)

In [54]:
print("Questions:", len(length_df))
print("Mean tokens:", round(length_df["max_tokens"].mean(), 1))
print("Median tokens:", round(length_df["max_tokens"].median(), 1))
print("95th percentile:", round(length_df["max_tokens"].quantile(0.95), 1))
print("Maximum tokens:", length_df["max_tokens"].max())

too_long = length_df[
    length_df["max_tokens"] > config.max_sequence_length
]

print(
    "Over 2048:",
    len(too_long),
    f"({len(too_long) / len(length_df) * 100:.2f}%)"
)

Questions: 1404
Mean tokens: 705.6
Median tokens: 511.5
95th percentile: 1695.7
Maximum tokens: 3129
Over 2048: 38 (2.71%)


In [55]:
too_long.sort_values(
    "max_tokens",
    ascending=False
).head(20)

,subject,test_index,min_tokens,max_tokens
1285,high_school_european_history,136,3129,3129
708,high_school_european_history,99,3096,3096
896,high_school_european_history,72,3025,3025
485,high_school_european_history,57,2932,2932
107,high_school_european_history,69,2929,2929
1215,high_school_european_history,25,2831,2831
1089,high_school_european_history,65,2825,2825
921,high_school_european_history,114,2809,2809
143,high_school_european_history,119,2756,2756
225,high_school_european_history,51,2722,2722


In [56]:
def choose_effective_ntrain(
    subject,
    dev_df,
    test_df,
    test_index,
    max_length=2048,
    requested_ntrain=5,
):
    """
    4 permutationすべてがmax_length以内に入る最大のshot数を返す。
    同一問題では4配置すべて同じshot数を使用する。
    """

    for ntrain in range(requested_ntrain, -1, -1):

        all_fit = True

        for shift in range(4):
            order = cyclic_order(shift)

            prompt = build_mmlu_prompt_5shot(
                subject=subject,
                dev_df=dev_df,
                test_df=test_df,
                test_index=test_index,
                order=order,
                ntrain=ntrain,
            )

            token_length = len(
                tokenizer.encode(prompt)
            )

            if token_length > max_length:
                all_fit = False
                break

        if all_fit:
            return ntrain

    raise ValueError(
        f"Even 0-shot exceeds {max_length} tokens: "
        f"{subject}[{test_index}]"
    )

In [57]:
effective_shots = []

for _, manifest_row in manifest.iterrows():

    subject = str(
        manifest_row["subject"]
    )

    test_index = int(
        manifest_row["test_index"]
    )

    dev_df = dev_cache[subject]
    test_df = test_cache[subject]

    ntrain = choose_effective_ntrain(
        subject=subject,
        dev_df=dev_df,
        test_df=test_df,
        test_index=test_index,
        max_length=config.max_sequence_length,
        requested_ntrain=5,
    )

    effective_shots.append({
        "subject": subject,
        "test_index": test_index,
        "effective_ntrain": ntrain,
    })

effective_shots_df = pd.DataFrame(
    effective_shots
)

print(
    effective_shots_df[
        "effective_ntrain"
    ].value_counts().sort_index(
        ascending=False
    )
)

effective_ntrain
5    1366
4      23
3      13
2       2
Name: count, dtype: int64


In [58]:
def evaluate_mmlu_question_5shot(
    subject,
    test_index,
    dev_df,
    test_df,
):
    # 4 permutationすべてが2048 token以内になる
    # 最大のshot数を使用
    effective_ntrain = choose_effective_ntrain(
        subject=subject,
        dev_df=dev_df,
        test_df=test_df,
        test_index=test_index,
        max_length=config.max_sequence_length,
        requested_ntrain=5,
    )

    permutation_probs = []
    mapped_probs = []
    prompt_lengths = []

    for shift in range(4):
        order = cyclic_order(shift)

        prompt = build_mmlu_prompt_5shot(
            subject=subject,
            dev_df=dev_df,
            test_df=test_df,
            test_index=test_index,
            order=order,
            ntrain=effective_ntrain,
        )

        prompt_lengths.append(
            len(tokenizer.encode(prompt))
        )

        # 表示上のA/B/C/D確率
        position_probs = get_choice_probs_array(prompt)
        permutation_probs.append(position_probs)

        # semantic optionへ戻す
        mapped = np.zeros(4)

        for position, original_index in enumerate(order):
            mapped[original_index] = position_probs[position]

        mapped_probs.append(mapped)

    # shift=0が通常配置
    baseline_probs = permutation_probs[0]

    baseline_pred = LABELS[
        np.argmax(baseline_probs)
    ]

    # semantic optionに戻した4配置の平均
    debiased_probs = np.mean(
        np.stack(mapped_probs),
        axis=0,
    )

    debiased_pred = LABELS[
        np.argmax(debiased_probs)
    ]

    row = test_df.iloc[test_index]
    label = str(row.iloc[-1]).strip()

    result = {
        "subject": subject,
        "test_index": int(test_index),

        "question": row.iloc[0],
        "A": row.iloc[1],
        "B": row.iloc[2],
        "C": row.iloc[3],
        "D": row.iloc[4],

        "label": label,

        "effective_ntrain": effective_ntrain,
        "max_prompt_tokens": max(prompt_lengths),

        "baseline_pred": baseline_pred,
        "baseline_correct": baseline_pred == label,

        "debiased_pred": debiased_pred,
        "debiased_correct": debiased_pred == label,
    }

    for j, name in enumerate(LABELS):
        result[f"baseline_{name}_prob"] = float(
            baseline_probs[j]
        )

        result[f"debiased_{name}_prob"] = float(
            debiased_probs[j]
        )

    # 各permutationについて表示位置確率も保存
    for shift in range(4):
        for j, name in enumerate(LABELS):
            result[f"perm{shift}_{name}_prob"] = float(
                permutation_probs[shift][j]
            )

    return result

In [59]:
OUTPUT_PATH_5SHOT = (
    OUTPUT_DIR
    / "aikengpt_mmlu_10pct_seed42_5shot.csv"
)

print(OUTPUT_PATH_5SHOT)

/content/drive/MyDrive/aikengpt_mmlu/aikengpt_mmlu_10pct_seed42_5shot.csv


In [60]:
import os
import pandas as pd

DATA_DIR = "/content/mmlu-reference/data"

dev_cache = {}
test_cache = {}

# 既に完了している問題
if OUTPUT_PATH_5SHOT.exists():
    existing_5shot = pd.read_csv(
        OUTPUT_PATH_5SHOT
    )

    completed = set(
        zip(
            existing_5shot["subject"].astype(str),
            existing_5shot["test_index"].astype(int),
        )
    )

    print(
        f"Already completed: {len(completed)}"
    )
else:
    completed = set()
    print("Starting 5-shot evaluation from scratch.")

TOTAL = len(manifest)

for _, manifest_row in manifest.iterrows():

    subject = str(
        manifest_row["subject"]
    )

    test_index = int(
        manifest_row["test_index"]
    )

    key = (subject, test_index)

    if key in completed:
        continue

    # dev
    if subject not in dev_cache:
        dev_path = os.path.join(
            DATA_DIR,
            "dev",
            f"{subject}_dev.csv",
        )

        dev_cache[subject] = pd.read_csv(
            dev_path,
            header=None,
        )

    # test
    if subject not in test_cache:
        test_path = os.path.join(
            DATA_DIR,
            "test",
            f"{subject}_test.csv",
        )

        test_cache[subject] = pd.read_csv(
            test_path,
            header=None,
        )

    dev_df = dev_cache[subject]
    test_df = test_cache[subject]

    print(
        f"Evaluating {len(completed)+1}/{TOTAL}: "
        f"{subject}[{test_index}]",
        flush=True,
    )

    try:
        result = evaluate_mmlu_question_5shot(
            subject=subject,
            test_index=test_index,
            dev_df=dev_df,
            test_df=test_df,
        )

        row_df = pd.DataFrame([result])

        file_exists = OUTPUT_PATH_5SHOT.exists()

        row_df.to_csv(
            OUTPUT_PATH_5SHOT,
            mode="a",
            header=not file_exists,
            index=False,
        )

        completed.add(key)

        print(
            f"  -> {result['effective_ntrain']}-shot "
            f"tokens={result['max_prompt_tokens']} "
            f"baseline={result['baseline_pred']} "
            f"debiased={result['debiased_pred']} "
            f"label={result['label']} "
            f"saved ({len(completed)}/{TOTAL})",
            flush=True,
        )

    except Exception as e:
        print()
        print(
            f"ERROR: {subject}[{test_index}]"
        )
        print(repr(e))
        raise

Starting 5-shot evaluation from scratch.
Evaluating 1/1404: high_school_macroeconomics[237]
  -> 5-shot tokens=398 baseline=C debiased=A label=A saved (1/1404)
Evaluating 2/1404: high_school_government_and_politics[143]
  -> 5-shot tokens=461 baseline=A debiased=D label=A saved (2/1404)
Evaluating 3/1404: high_school_chemistry[72]
  -> 5-shot tokens=490 baseline=C debiased=C label=D saved (3/1404)
Evaluating 4/1404: sociology[154]
  -> 5-shot tokens=445 baseline=B debiased=A label=D saved (4/1404)
Evaluating 5/1404: high_school_government_and_politics[140]
  -> 5-shot tokens=463 baseline=B debiased=B label=A saved (5/1404)
Evaluating 6/1404: security_studies[62]
  -> 5-shot tokens=1271 baseline=D debiased=C label=D saved (6/1404)
Evaluating 7/1404: moral_disputes[95]
  -> 5-shot tokens=460 baseline=A debiased=B label=A saved (7/1404)
Evaluating 8/1404: miscellaneous[455]
  -> 5-shot tokens=304 baseline=B debiased=C label=B saved (8/1404)
Evaluating 9/1404: miscellaneous[87]
  -> 5-shot

In [61]:
five_df = pd.read_csv(
    OUTPUT_PATH_5SHOT
)

print("Rows:", len(five_df))

print(
    "5-shot baseline:",
    five_df["baseline_correct"].mean()
)

print(
    "5-shot debiased:",
    five_df["debiased_correct"].mean()
)

print("\nEffective shots:")
print(
    five_df["effective_ntrain"]
    .value_counts()
    .sort_index(ascending=False)
)

Rows: 1404
5-shot baseline: 0.2492877492877493
5-shot debiased: 0.26495726495726496

Effective shots:
effective_ntrain
5    1366
4      23
3      13
2       2
Name: count, dtype: int64


In [62]:
print(
    f"0-shot baseline : "
    f"{final_df['baseline_correct'].mean()*100:.2f}%"
)

print(
    f"0-shot debiased : "
    f"{final_df['debiased_correct'].mean()*100:.2f}%"
)

print(
    f"5-shot baseline : "
    f"{five_df['baseline_correct'].mean()*100:.2f}%"
)

print(
    f"5-shot debiased : "
    f"{five_df['debiased_correct'].mean()*100:.2f}%"
)

0-shot baseline : 26.07%
0-shot debiased : 25.57%
5-shot baseline : 24.93%
5-shot debiased : 26.50%


In [63]:
changed_5 = (
    five_df["baseline_pred"]
    != five_df["debiased_pred"]
)

became_correct_5 = (
    (~five_df["baseline_correct"])
    & five_df["debiased_correct"]
)

became_wrong_5 = (
    five_df["baseline_correct"]
    & (~five_df["debiased_correct"])
)

print(
    "Prediction changed:",
    changed_5.sum(),
    f"({changed_5.mean()*100:.2f}%)"
)

print(
    "Wrong -> Correct:",
    became_correct_5.sum()
)

print(
    "Correct -> Wrong:",
    became_wrong_5.sum()
)

print(
    "Net gain:",
    became_correct_5.sum()
    - became_wrong_5.sum()
)

Prediction changed: 993 (70.73%)
Wrong -> Correct: 263
Correct -> Wrong: 241
Net gain: 22


In [64]:
print("5-shot displayed-position probabilities:")

for label in ["A", "B", "C", "D"]:
    cols = [
        f"perm0_{label}_prob",
        f"perm1_{label}_prob",
        f"perm2_{label}_prob",
        f"perm3_{label}_prob",
    ]

    mean_prob = five_df[cols].to_numpy().mean()

    print(
        f"{label}: {mean_prob:.6f}"
    )

5-shot displayed-position probabilities:
A: 0.230472
B: 0.253595
C: 0.289657
D: 0.226276
